# Lektion 11 - Model Context Protocol (MCP)

**Model Context Protocol (MCP)** er en åben standard, der gør det muligt for agenter dynamisk at opdage og bruge værktøjer, ressourcer og datakilder i runtime. I stedet for at hardkode værktøjer i en agent, lader MCP agenter forbinde til eksterne servere, der eksponerer kapaciteter efter behov.

I denne lektion vil du lære:
- Hvad MCP er, og hvorfor det er vigtigt for agentsystemer
- Hvordan MCP klient-server arkitekturen fungerer
- Hvordan man bygger agenter, der bruger MCP-stil værktøjsopdagelse


## Opsætning

**Forudsætninger:**
- Microsoft Foundry-projekt med en implementeret model
- Kør `az login` for autentificering


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Hvad er Model Context Protocol (MCP)?

MCP definerer en standard måde, hvorpå AI-agenter kan opdage og interagere med eksterne værktøjer og datakilder:

- **MCP Server**: Eksponerer værktøjer, ressourcer og prompts via en standardprotokol
- **MCP Client**: Agent-runtime, der opretter forbindelse til servere og opdager tilgængelige kapaciteter
- **Dynamisk opdagelse**: Agenter behøver ikke hardkodede værktøjer — de opdager, hvad der er tilgængeligt under kørsel

Dette er kraftfuldt til at bygge udvidelige agentsystemer, hvor nye kapaciteter kan tilføjes uden at ændre agentkoden.


## Hvordan MCP fungerer

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agenten (MCP-klient) opretter forbindelse til en MCP-server
2. Serveren svarer med en liste over tilgængelige værktøjer og deres skemaer
3. Agenten kan derefter kalde ethvert opdaget værktøj under sin ræsonnering
4. Resultaterne flyder tilbage gennem samme protokol


## Simulering af MCP-værktøjsopdagelse

Da en rigtig MCP-server kræver en kørende serverproces, vil vi demonstrere mønstret ved hjælp af `@tool` funktioner, der simulerer, hvad en MCP-forbundet indkvarteringstjeneste ville levere.

I produktionen ville disse værktøjer blive opdaget dynamisk fra en MCP-server i stedet for at blive defineret lokalt.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Bygge en agent med MCP-stil værktøjer


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP i produktion

I et produktionsmiljø muliggør MCP kraftfulde mønstre:

- **Dynamisk værktøjsopdagelse**: Agenter opretter forbindelse til MCP-servere og opdager værktøjer ved runtime
- **Adskilt arkitektur**: Værktøjsudbydere kan opdatere uafhængigt af agenten
- **Deling på tværs af organisationer**: Teams kan eksponere kapaciteter via MCP-servere, som enhver agent kan bruge
- **Support til Microsoft Agent Framework**: MAF har indbygget MCP-klientunderstøttelse via `mcp` integrationen

For at bruge en rigtig MCP-server med MAF skal du oprette forbindelse via `hosted_mcp_tool()` eller MCP-klientintegration.

**Lær mere:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Resumé

I denne lektion lærte du:
- **MCP** er en åben standard for dynamisk værktøjsopdagelse mellem agenter og værktøjsudbydere
- Den **klient-server arkitektur** gør det muligt for agenter at opdage funktioner under kørsel
- MCP muliggør **udvidelige, løst koblede agentsystemer**, hvor værktøjer kan tilføjes uden kodeændringer
- Microsoft Agent Framework tilbyder **indbygget MCP-understøttelse** til produktionsbrug


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
